# Graph-view data generation (GNN dataset) — Colab runner

Runs `generate_graph.py`: same 7 scenarios as the tabular generator, exported as a full-network graph view (static topology + per-timestep node/edge tensors). Lives on the **graph-export** branch. **Edit `REPO_URL` / `BRANCH`.**

In [ ]:
import os
import getpass

REPO_OWNER = "savvy-sam"
REPO_NAME = "sewer-blockage"

PAT = os.environ.get("GITHUB_PAT") or getpass.getpass("Enter your GitHub PAT: ")
REPO_URL = f"https://{PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

name = REPO_NAME

if os.path.isdir(name):
    !cd {name} && git remote set-url origin {REPO_URL} && git pull --ff-only
else:
    !git clone {REPO_URL}

%cd {name}

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import os, generate_graph
print('generate_graph.py:', os.path.getsize('generate_graph.py'), 'bytes', '| has main():', hasattr(generate_graph, 'main'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Generate

Start small: the graph view reads ~2000 SWMM values per timestep, so it is slower than the tabular run. Keep `--graph-step-min` at 5+ and a small `--n-per-scenario` first. Use the **same `--seed` / `--scenarios`** as your tabular run so the two datasets stay aligned (identical underlying simulations).

In [ ]:
!python generate_graph.py --inp inputs/BellingeSWMM_v021_nopervious.inp --targets inputs/blockage_targets.csv --out graph_data --scenarios 1,2,3,4,5,6,7 --n-per-scenario 3 --graph-step-min 5 --seed 0

## Inspect the output

In [ ]:
import numpy as np, glob, os
gs = np.load('graph_data/graph_static.npz', allow_pickle=True)
print('static graph:  node_static', gs['node_static'].shape, '| edge_index', gs['edge_index'].shape, '| edge_static', gs['edge_static'].shape)
f = sorted(glob.glob('graph_data/runs/*.npz'))[0]
d = np.load(f, allow_pickle=True)
print('example run:', os.path.basename(f))
for k in d.files:
    v = d[k]
    print('  %-18s' % k, v.shape if hasattr(v, 'ndim') and v.ndim else v)
print('  label counts:', dict(zip(*[x.tolist() for x in np.unique(d['y_global'], return_counts=True)])))
print('  observed sensor nodes:', int(d['sensor_node_mask'].sum()))

In [ ]:
!cp -r graph_data '/content/drive/MyDrive/sewer_graph_data'
print('saved -> MyDrive/sewer_graph_data')